# Notebook 7 — Encode Geospatial Network into the CANOE Schema

This notebook converts the geospatial outputs developed in previous notebooks into a CANOE-compatible SQLite database.

The objective is to replace the synthetic grid-neighbor representation used in the prototype model with a transport network derived from the Canadian basemap and road connectivity analysis while preserving compatibility with the existing CANOE/TEMOA model structure.

Transport links are represented using CANOE pseudo-regions of the form

```
region_from-region_to
```

where the two regions correspond to adjacent basemap polygons connected by existing infrastructure. This forms a dual graph representation of Canadian infrastructure layers aligned under a specified spatial resolution.

Initially, this notebook focuses on road-based transport technologies and encodes only links for which road connectivity has been identified. The absence of a link implies that transport between those regions is infeasible.

Rather than rebuilding the complete database from raw CSV files, this notebook loads the existing CANOE database and performs a schema reconciliation step. Inherited tables are filtered to the geospatial region topology and augmented with new transport technologies, producing a functional geospatial test database suitable for MILP execution.

The resulting database provides an intermediate development layer between the geospatial preprocessing workflow and eventual integration into the core CANOE modules.

---

## Inputs

### Basemap regions

From Notebook 4:

* Regional polygon geometries
* Region identifiers
* Region centroids

### Neighbor relationships

From Notebook 5:

* Polygon adjacency graph
* Neighbor pairs
* Inter-region distances

### Road connectivity

From Notebook 6:

* Weak road connectivity
* Strong road connectivity

### Existing CANOE database

* Existing CANOE SQLite database
* Schema definitions
* Technology definitions
* Commodity definitions
* Supporting tables

---

## Outputs

This notebook modifies and validates CANOE tables including:

* Region
* Technology
* Efficiency
* CostVariable
* CostInvest
* ETLSegment
* Demand
* LimitCapacity
* Supporting schema tables

and exports complete SQLite databases suitable for direct use by the CANOE/TEMOA solver.

---

## Conceptual workflow

1. Load the regional basemap and road connectivity outputs.
2. Load the existing CANOE database.
3. Replace the synthetic region representation with geospatial regions.
4. Build transport edges from connected neighboring regions.
5. Encode transport technologies for each valid edge.
6. Reconcile inherited node and edge tables with the geospatial topology.
7. Validate schema consistency.
8. Export SQLite databases.
9. Test MILP execution using the existing CANOE workflow.

This notebook serves as a graph-to-schema encoder and schema reconciliation layer between the geospatial preprocessing workflow and eventual integration into the core CANOE modules.

In [1]:
# =============================================================================
# Dependencies
# =============================================================================

from pathlib import Path

import sqlite3

import numpy as np
import pandas as pd

import geopandas as gpd

import matplotlib.pyplot as plt

import db_mgmt

In [2]:
# =============================================================================
# Project directories
# =============================================================================

PROJECT_ROOT = Path.cwd().parent

DATA_FILES = PROJECT_ROOT / "data_files"

RAW_BASEMAPS = DATA_FILES / "raw" / "basemaps"

PROCESSED_BASEMAPS = DATA_FILES / "processed" / "basemaps"
PROCESSED_GRAPH = DATA_FILES / "processed" / "graph"
PROCESSED_ROAD_CONNECTIVITY = DATA_FILES / "processed" / "road_connectivity"
PROCESSED_SCHEMA = DATA_FILES / "processed" / "schema"

PROCESSED_SCHEMA.mkdir(
    parents=True,
    exist_ok=True,
)

In [3]:
# =============================================================================
# Input files
# =============================================================================

RAW_BASEMAP_PATH = RAW_BASEMAPS / "lpr_000b21a_e.shp"

GRAPH_NODE_PATH = (
    PROCESSED_GRAPH
    / "canada_basemap_1deg_centroid_graph_nodes.gpkg"
)

GRAPH_EDGE_PATH = (
    PROCESSED_GRAPH
    / "canada_basemap_1deg_centroid_graph_edges.csv"
)

ROAD_EDGE_CONNECTIONS_WEAK_PATH = (
    PROCESSED_ROAD_CONNECTIVITY
    / "CANADA_filtered_road_networks__canada_basemap_1deg_centroid_graph_nodes_weak_road_edge_connections.csv"
)

ROAD_EDGES_WEAK_GPKG_PATH = (
    PROCESSED_ROAD_CONNECTIVITY
    / "CANADA_filtered_road_networks__canada_basemap_1deg_centroid_graph_nodes_weak_road_edges.gpkg"
)

RAW_SCHEMA_PATH = DATA_FILES / "canoe_dataset_schema.sql"

BASELINE_SQLITE_PATH = DATA_FILES / "CANOE_geospatial.sqlite"

In [4]:
# =============================================================================
# Output files
# =============================================================================

OUTPUT_SQLITE_WEAK_PATH = (
    PROCESSED_SCHEMA
    / "CANOE_geospatial_1deg_graph_roads_weak.sqlite"
)

OUTPUT_SQLITE_STRONG_PATH = (
    PROCESSED_SCHEMA
    / "CANOE_geospatial_1deg_graph_roads_strong.sqlite"
)

In [5]:
# =============================================================================
# Validate input files
# =============================================================================

input_paths = {
    "raw_basemap": RAW_BASEMAP_PATH,
    "graph_nodes": GRAPH_NODE_PATH,
    "graph_edges": GRAPH_EDGE_PATH,
    "road_edge_connections_weak": ROAD_EDGE_CONNECTIONS_WEAK_PATH,
    "road_edges_weak_gpkg": ROAD_EDGES_WEAK_GPKG_PATH,
    "raw_schema": RAW_SCHEMA_PATH,
    "baseline_sqlite": BASELINE_SQLITE_PATH,
}

missing_paths = {
    name: path
    for name, path in input_paths.items()
    if not path.exists()
}

if missing_paths:
    for name, path in missing_paths.items():
        print(f"Missing {name}: {path}")
    raise FileNotFoundError("One or more required input files are missing.")

print("All required input files found.")

All required input files found.


In [6]:
# =============================================================================
# Load graph inputs
# =============================================================================

graph_nodes = gpd.read_file(
    GRAPH_NODE_PATH
)

graph_edges = pd.read_csv(
    GRAPH_EDGE_PATH
)

road_edge_connections_weak = pd.read_csv(
    ROAD_EDGE_CONNECTIONS_WEAK_PATH
)

road_edges_weak_gdf = gpd.read_file(
    ROAD_EDGES_WEAK_GPKG_PATH
)

print(f"Graph nodes: {len(graph_nodes):,} rows")
print(f"Graph edges: {len(graph_edges):,} rows")
print(f"Weak road edge connections: {len(road_edge_connections_weak):,} rows")
print(f"Weak road edge geometries: {len(road_edges_weak_gdf):,} rows")

print("\nCRS:")

print(f"Graph nodes: {graph_nodes.crs}")
print(f"Weak road edge geometries: {road_edges_weak_gdf.crs}")

Graph nodes: 1,692 rows
Graph edges: 5,886 rows
Weak road edge connections: 5,886 rows
Weak road edge geometries: 1,518 rows

CRS:
Graph nodes: EPSG:4326
Weak road edge geometries: EPSG:4326


In [7]:
# =============================================================================
# Load baseline CANOE database
# =============================================================================

db = db_mgmt.sqlite_to_dfs(
    BASELINE_SQLITE_PATH,
)

print(f"{len(db)} tables loaded.")

85 tables loaded.


In [8]:
# =============================================================================
# Inspect available tables
# =============================================================================

sorted(db.keys())

['CapacityCredit',
 'CapacityFactorProcess',
 'CapacityFactorTech',
 'CapacityToActivity',
 'Commodity',
 'CommodityType',
 'ConstructionInput',
 'CostEmission',
 'CostFixed',
 'CostInvest',
 'CostVariable',
 'DataQualityCredibility',
 'DataQualityGeography',
 'DataQualityStructure',
 'DataQualityTechnology',
 'DataQualityTime',
 'DataSet',
 'DataSource',
 'Demand',
 'DemandSpecificDistribution',
 'ETLSegment',
 'Efficiency',
 'EfficiencyVariable',
 'EmissionActivity',
 'EmissionEmbodied',
 'EmissionEndOfLife',
 'EndOfLifeOutput',
 'ExistingCapacity',
 'LifetimeProcess',
 'LifetimeSurvivalCurve',
 'LifetimeTech',
 'LimitActivity',
 'LimitActivityShare',
 'LimitAnnualCapacityFactor',
 'LimitCapacity',
 'LimitCapacityShare',
 'LimitDegrowthCapacity',
 'LimitDegrowthNewCapacity',
 'LimitDegrowthNewCapacityDelta',
 'LimitEmission',
 'LimitGrowthCapacity',
 'LimitGrowthNewCapacity',
 'LimitGrowthNewCapacityDelta',
 'LimitNewCapacity',
 'LimitNewCapacityShare',
 'LimitResource',
 'LimitSeaso

In [9]:
# =============================================================================
# Inspect core schema tables
# =============================================================================

core_tables = [
    "Region",
    "Technology",
    "TechnologyType",
    "Commodity",
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
    "LimitCapacity",
    "LimitNewCapacity",
    "ExistingCapacity",
    "DataSet",
    "DataSource",
]

for table in core_tables:
    df = db[table]
    print(f"\n{table}")
    print(f"  rows: {len(df):,}")
    print(f"  columns: {list(df.columns)}")


Region
  rows: 2,259
  columns: ['region', 'notes']

Technology
  rows: 12
  columns: ['tech', 'flag', 'sector', 'category', 'sub_category', 'unlim_cap', 'annual', 'reserve', 'curtail', 'retire', 'flex', 'exchange', 'seas_stor', 'description', 'data_id']

TechnologyType
  rows: 4
  columns: ['label', 'description']

Commodity
  rows: 7
  columns: ['name', 'flag', 'description', 'data_id']

Efficiency
  rows: 62,188
  columns: ['region', 'input_comm', 'tech', 'vintage', 'output_comm', 'efficiency', 'notes', 'data_source', 'dq_cred', 'dq_geog', 'dq_struc', 'dq_tech', 'dq_time', 'data_id']

CostVariable
  rows: 50,487
  columns: ['region', 'period', 'tech', 'vintage', 'cost', 'units', 'notes', 'data_source', 'dq_cred', 'dq_geog', 'dq_struc', 'dq_tech', 'dq_time', 'data_id']

CostInvest
  rows: 4,518
  columns: ['region', 'tech', 'vintage', 'cost', 'units', 'notes', 'data_source', 'dq_cred', 'dq_geog', 'dq_struc', 'dq_tech', 'dq_time', 'data_id']

ETLSegment
  rows: 193,552
  columns: ['r

In [10]:
# =============================================================================
# Inspect baseline technologies and commodities
# =============================================================================

display(
    db["Technology"].sort_values("tech")
)

display(
    db["TechnologyType"].sort_values("label")
)

display(
    db["Commodity"].sort_values("name")
)

,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
2,CO2_CAP,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
7,CO2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
0,ELC_GEN,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
5,ELC_TRANS,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
11,GSL_BACKUP,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
10,GSL_DEMAND,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
9,GSL_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
4,GSL_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
6,H2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
1,H2_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001


,label,description
2,p,production
0,pb,baseload production technology
1,ps,storage production technology
3,t,transport


,name,flag,description,data_id
1,ch3oh,wa,methanol,None
2,co2,wa,co2 captured,None
5,d_gsl,d,gasoline demand,None
4,elc,wa,electricity,None
6,ethos,s,dummy,None
3,gsl,wa,gasoline,None
0,h2,wa,hydrogen,None


In [11]:
# =============================================================================
# Inspect baseline technology definitions
# =============================================================================

technology = db["Technology"].copy()

display(
    technology.sort_values(
        "tech"
    ).reset_index(
        drop=True
    )
)

,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
0,CO2_CAP,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
1,CO2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
2,ELC_GEN,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
3,ELC_TRANS,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
4,GSL_BACKUP,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
5,GSL_DEMAND,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
6,GSL_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
7,GSL_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
8,H2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
9,H2_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001


In [12]:
# =============================================================================
# Inspect baseline regions
# =============================================================================

region = db["Region"].copy()

print(f"Number of regions: {len(region):,}")

display(
    region.head(20)
)

Number of regions: 2,259


,region,notes
0,R0,None
1,R1,None
2,R2,None
3,R3,None
4,R4,None
5,R5,None
6,R6,None
7,R7,None
8,R8,None
9,R9,None


In [13]:
# =============================================================================
# Inspect graph identifiers
# =============================================================================

print(f"Graph nodes: {len(graph_nodes):,}")
print(f"Graph edges: {len(graph_edges):,}")
print(f"Weak road edge connections: {len(road_edge_connections_weak):,}")

display(
    graph_nodes.head()
)

display(
    graph_edges.head()
)

display(
    road_edge_connections_weak.head()
)

Graph nodes: 1,692
Graph edges: 5,886
Weak road edge connections: 5,886


,lon_min,lon_max,lat_min,lat_max,lon,lat,region,site_id,resolution_deg,keep_method,up_id,down_id,right_id,left_id,up_distance,down_distance,right_distance,left_distance,n_neighbors,geometry
0,-82.0,-81.0,43.0,44.0,-81.5,43.5,R0,R0,1.0,centroid,R-999,R-999,R1,R-999,NaN,NaN,80.876192,NaN,1,"POLYGON ((-81 43, -81 44, -82 44, -82 43, -81 ..."
1,-81.0,-80.0,43.0,44.0,-80.5,43.5,R1,R1,1.0,centroid,R3,R-999,R-999,R0,111.112243,NaN,NaN,80.876192,2,"POLYGON ((-80 43, -80 44, -81 44, -81 43, -80 ..."
2,-66.0,-65.0,43.0,44.0,-65.5,43.5,R2,R2,1.0,centroid,R8,R-999,R-999,R-999,111.112243,NaN,NaN,NaN,1,"POLYGON ((-65 43, -65 44, -66 44, -66 43, -65 ..."
3,-81.0,-80.0,44.0,45.0,-80.5,44.5,R3,R3,1.0,centroid,R-999,R1,R4,R-999,NaN,111.112243,79.529066,NaN,2,"POLYGON ((-80 44, -80 45, -81 45, -81 44, -80 ..."
4,-80.0,-79.0,44.0,45.0,-79.5,44.5,R4,R4,1.0,centroid,R10,R-999,R5,R3,111.131778,NaN,79.529066,79.529066,3,"POLYGON ((-79 44, -79 45, -80 45, -80 44, -79 ..."


,edge_region,region_from,region_to,direction,lon_from,lat_from,lon_to,lat_to,distance_km,resolution_deg
0,R0-R1,R0,R1,right,-81.5,43.5,-80.5,43.5,80.876192,1.0
1,R1-R3,R1,R3,up,-80.5,43.5,-80.5,44.5,111.112243,1.0
2,R1-R0,R1,R0,left,-80.5,43.5,-81.5,43.5,80.876192,1.0
3,R2-R8,R2,R8,up,-65.5,43.5,-65.5,44.5,111.112243,1.0
4,R3-R1,R3,R1,down,-80.5,44.5,-80.5,43.5,111.112243,1.0


,edge_region,region_from,region_to,direction,lon_from,lat_from,lon_to,lat_to,distance_km,resolution_deg,delta_lon,delta_lat,has_road_from,has_road_to,has_road_connection,connection_method,region_pair
0,R0-R1,R0,R1,right,-81.5,43.5,-80.5,43.5,80.876192,1.0,1.0,0.0,True,True,True,weak_node_presence_adjacency,R0-R1
1,R1-R3,R1,R3,up,-80.5,43.5,-80.5,44.5,111.112243,1.0,0.0,1.0,True,True,True,weak_node_presence_adjacency,R1-R3
2,R1-R0,R1,R0,left,-80.5,43.5,-81.5,43.5,80.876192,1.0,-1.0,0.0,True,True,True,weak_node_presence_adjacency,R1-R0
3,R2-R8,R2,R8,up,-65.5,43.5,-65.5,44.5,111.112243,1.0,0.0,1.0,True,True,True,weak_node_presence_adjacency,R2-R8
4,R3-R1,R3,R1,down,-80.5,44.5,-80.5,43.5,111.112243,1.0,0.0,-1.0,True,True,True,weak_node_presence_adjacency,R3-R1


In [14]:
# =============================================================================
# Build CANOE Region table from graph nodes
# =============================================================================

region_table = (
    graph_nodes[["region"]]
    .drop_duplicates()
    .sort_values(
        "region",
        key=lambda s: s.str.extract(r"R(\d+)")[0].astype(int),
    )
    .reset_index(drop=True)
)

region_table["notes"] = (
    "1 degree CANOE geospatial graph node retained by centroid method"
)

print(f"New Region table rows: {len(region_table):,}")
print(f"Unique regions: {region_table['region'].nunique():,}")

display(region_table.head())
display(region_table.tail())

New Region table rows: 1,692
Unique regions: 1,692


,region,notes
0,R0,1 degree CANOE geospatial graph node retained ...
1,R1,1 degree CANOE geospatial graph node retained ...
2,R2,1 degree CANOE geospatial graph node retained ...
3,R3,1 degree CANOE geospatial graph node retained ...
4,R4,1 degree CANOE geospatial graph node retained ...


,region,notes
1687,R1687,1 degree CANOE geospatial graph node retained ...
1688,R1688,1 degree CANOE geospatial graph node retained ...
1689,R1689,1 degree CANOE geospatial graph node retained ...
1690,R1690,1 degree CANOE geospatial graph node retained ...
1691,R1691,1 degree CANOE geospatial graph node retained ...


In [15]:
# =============================================================================
# Validate geospatial Region table
# =============================================================================

assert "R-999" not in set(region_table["region"]), (
    "R-999 should not be a real region."
)

assert (
    len(region_table) == region_table["region"].nunique()
), "Duplicate region IDs found."

assert (
    len(region_table) == len(graph_nodes)
), "Region table row count does not match graph node row count."

print("Region table validated.")

Region table validated.


In [16]:
# =============================================================================
# Validate 1-degree centroid coordinate convention
# =============================================================================

resolution = graph_nodes["resolution_deg"].unique()

print(f"Resolution values: {resolution}")

assert (
    len(resolution) == 1
), "Multiple resolutions found in graph node table."

resolution_deg = float(resolution[0])

lon_offset = (
    (
        graph_nodes["lon"]
        - graph_nodes["lon_min"]
    )
    / resolution_deg
).round(6)

lat_offset = (
    (
        graph_nodes["lat"]
        - graph_nodes["lat_min"]
    )
    / resolution_deg
).round(6)

assert (
    set(lon_offset.unique()) == {0.5}
), "Longitude centroids are not centered within cells."

assert (
    set(lat_offset.unique()) == {0.5}
), "Latitude centroids are not centered within cells."

print("Centroid coordinate convention validated.")

Resolution values: [1.]
Centroid coordinate convention validated.


In [17]:
# =============================================================================
# Inspect weak road edge tables
# =============================================================================

print(f"Weak road edge connection rows: {len(road_edge_connections_weak):,}")
print(f"Weak road edge geometry rows: {len(road_edges_weak_gdf):,}")

print("\nConnection columns:")
print(list(road_edge_connections_weak.columns))

print("\nGeometry columns:")
print(list(road_edges_weak_gdf.columns))

display(
    road_edge_connections_weak.head()
)

display(
    road_edges_weak_gdf.head()
)

Weak road edge connection rows: 5,886
Weak road edge geometry rows: 1,518

Connection columns:
['edge_region', 'region_from', 'region_to', 'direction', 'lon_from', 'lat_from', 'lon_to', 'lat_to', 'distance_km', 'resolution_deg', 'delta_lon', 'delta_lat', 'has_road_from', 'has_road_to', 'has_road_connection', 'connection_method', 'region_pair']

Geometry columns:
['edge_region', 'region_from', 'region_to', 'direction', 'lon_from', 'lat_from', 'lon_to', 'lat_to', 'distance_km', 'resolution_deg', 'delta_lon', 'delta_lat', 'has_road_from', 'has_road_to', 'has_road_connection', 'connection_method', 'region_pair', 'geometry']


,edge_region,region_from,region_to,direction,lon_from,lat_from,lon_to,lat_to,distance_km,resolution_deg,delta_lon,delta_lat,has_road_from,has_road_to,has_road_connection,connection_method,region_pair
0,R0-R1,R0,R1,right,-81.5,43.5,-80.5,43.5,80.876192,1.0,1.0,0.0,True,True,True,weak_node_presence_adjacency,R0-R1
1,R1-R3,R1,R3,up,-80.5,43.5,-80.5,44.5,111.112243,1.0,0.0,1.0,True,True,True,weak_node_presence_adjacency,R1-R3
2,R1-R0,R1,R0,left,-80.5,43.5,-81.5,43.5,80.876192,1.0,-1.0,0.0,True,True,True,weak_node_presence_adjacency,R1-R0
3,R2-R8,R2,R8,up,-65.5,43.5,-65.5,44.5,111.112243,1.0,0.0,1.0,True,True,True,weak_node_presence_adjacency,R2-R8
4,R3-R1,R3,R1,down,-80.5,44.5,-80.5,43.5,111.112243,1.0,0.0,-1.0,True,True,True,weak_node_presence_adjacency,R3-R1


,edge_region,region_from,region_to,direction,lon_from,lat_from,lon_to,lat_to,distance_km,resolution_deg,delta_lon,delta_lat,has_road_from,has_road_to,has_road_connection,connection_method,region_pair,geometry
0,R0-R1,R0,R1,right,-81.5,43.5,-80.5,43.5,80.876192,1.0,1.0,0.0,True,True,True,weak_node_presence_adjacency,R0-R1,"LINESTRING (-81.5 43.5, -80.5 43.5)"
1,R1-R3,R1,R3,up,-80.5,43.5,-80.5,44.5,111.112243,1.0,0.0,1.0,True,True,True,weak_node_presence_adjacency,R1-R3,"LINESTRING (-80.5 43.5, -80.5 44.5)"
2,R1-R0,R1,R0,left,-80.5,43.5,-81.5,43.5,80.876192,1.0,-1.0,0.0,True,True,True,weak_node_presence_adjacency,R1-R0,"LINESTRING (-80.5 43.5, -81.5 43.5)"
3,R2-R8,R2,R8,up,-65.5,43.5,-65.5,44.5,111.112243,1.0,0.0,1.0,True,True,True,weak_node_presence_adjacency,R2-R8,"LINESTRING (-65.5 43.5, -65.5 44.5)"
4,R3-R1,R3,R1,down,-80.5,44.5,-80.5,43.5,111.112243,1.0,0.0,-1.0,True,True,True,weak_node_presence_adjacency,R3-R1,"LINESTRING (-80.5 44.5, -80.5 43.5)"


In [18]:
# =============================================================================
# Build clean weak road-link table
# =============================================================================

road_links_weak = (
    road_edge_connections_weak
    .loc[
        road_edge_connections_weak["has_road_connection"]
    ]
    .copy()
)

road_links_weak = road_links_weak[
    [
        "edge_region",
        "region_from",
        "region_to",
        "direction",
        "connection_method",
        "distance_km",
        "lon_from",
        "lat_from",
        "lon_to",
        "lat_to",
    ]
].copy()

road_links_weak = (
    road_links_weak
    .drop_duplicates(
        subset=["edge_region"]
    )
    .reset_index(drop=True)
)

road_links_weak["canoe_region"] = road_links_weak["edge_region"]

print(f"Weak road links: {len(road_links_weak):,}")
print(f"Unique CANOE transport regions: {road_links_weak['canoe_region'].nunique():,}")

display(
    road_links_weak.head()
)

Weak road links: 1,518
Unique CANOE transport regions: 1,518


,edge_region,region_from,region_to,direction,connection_method,distance_km,lon_from,lat_from,lon_to,lat_to,canoe_region
0,R0-R1,R0,R1,right,weak_node_presence_adjacency,80.876192,-81.5,43.5,-80.5,43.5,R0-R1
1,R1-R3,R1,R3,up,weak_node_presence_adjacency,111.112243,-80.5,43.5,-80.5,44.5,R1-R3
2,R1-R0,R1,R0,left,weak_node_presence_adjacency,80.876192,-80.5,43.5,-81.5,43.5,R1-R0
3,R2-R8,R2,R8,up,weak_node_presence_adjacency,111.112243,-65.5,43.5,-65.5,44.5,R2-R8
4,R3-R1,R3,R1,down,weak_node_presence_adjacency,111.112243,-80.5,44.5,-80.5,43.5,R3-R1


In [19]:
# =============================================================================
# Validate weak road-link table
# =============================================================================

valid_regions = set(region_table["region"])

invalid_from = sorted(
    set(road_links_weak["region_from"]) - valid_regions
)

invalid_to = sorted(
    set(road_links_weak["region_to"]) - valid_regions
)

assert not invalid_from, f"Invalid region_from values found: {invalid_from[:10]}"
assert not invalid_to, f"Invalid region_to values found: {invalid_to[:10]}"

assert (
    road_links_weak["canoe_region"].str.contains("-", regex=False).all()
), "All CANOE transport regions should use region_from-region_to format."

assert (
    road_links_weak["canoe_region"].nunique() == len(road_links_weak)
), "Duplicate CANOE transport region IDs found."

assert (
    road_links_weak["canoe_region"].equals(road_links_weak["edge_region"])
), "canoe_region should match canonical edge_region."

print("Weak road-link table validated.")

Weak road-link table validated.


In [20]:
# =============================================================================
# Validate weak road-link distances
# =============================================================================

assert "distance_km" in road_links_weak.columns, (
    "road_links_weak should inherit distance_km from graph_edges."
)

assert road_links_weak["distance_km"].notna().all(), (
    "Road-link distances contain missing values."
)

assert (road_links_weak["distance_km"] > 0).all(), (
    "Road-link distances must be positive."
)

print("Weak road-link distance summary:")

display(
    road_links_weak["distance_km"].describe()
)

display(
    road_links_weak.head()
)

Weak road-link distance summary:


count    1518.000000
mean       87.027498
std        22.981981
min        42.721895
25%        67.909691
50%        76.762081
75%       111.267347
max       111.535694
Name: distance_km, dtype: float64

,edge_region,region_from,region_to,direction,connection_method,distance_km,lon_from,lat_from,lon_to,lat_to,canoe_region
0,R0-R1,R0,R1,right,weak_node_presence_adjacency,80.876192,-81.5,43.5,-80.5,43.5,R0-R1
1,R1-R3,R1,R3,up,weak_node_presence_adjacency,111.112243,-80.5,43.5,-80.5,44.5,R1-R3
2,R1-R0,R1,R0,left,weak_node_presence_adjacency,80.876192,-80.5,43.5,-81.5,43.5,R1-R0
3,R2-R8,R2,R8,up,weak_node_presence_adjacency,111.112243,-65.5,43.5,-65.5,44.5,R2-R8
4,R3-R1,R3,R1,down,weak_node_presence_adjacency,111.112243,-80.5,44.5,-80.5,43.5,R3-R1


In [21]:
# =============================================================================
# Inspect transport edge regions in baseline model
# =============================================================================

transport_regions = (
    pd.Series(
        db["CostVariable"]["region"].unique(),
        name="region",
    )
)

transport_regions = (
    transport_regions[
        transport_regions.str.contains(
            "-",
            regex=False,
            na=False,
        )
    ]
    .sort_values()
    .reset_index(drop=True)
)

print(
    f"Transport edge regions in baseline model: "
    f"{len(transport_regions):,}"
)

display(
    transport_regions.head(20)
)

Transport edge regions in baseline model: 8,774


0           R0-R1
1          R0-R11
2           R1-R0
3          R1-R12
4           R1-R2
5         R10-R21
6          R10-R9
7       R100-R101
8       R100-R118
9        R100-R85
10       R100-R99
11    R1000-R1001
12    R1000-R1032
13     R1000-R969
14     R1000-R999
15    R1001-R1000
16    R1001-R1002
17    R1001-R1033
18     R1001-R970
19    R1002-R1001
Name: region, dtype: object

In [22]:
# =============================================================================
# Define graph region sets by transport layer
# =============================================================================

VALID_NODE_REGIONS = set(region_table["region"])

VALID_PIPELINE_EDGE_REGIONS = set(graph_edges["edge_region"])

VALID_ROAD_EDGE_REGIONS = set(road_links_weak["canoe_region"])

print(f"Valid node regions: {len(VALID_NODE_REGIONS):,}")
print(f"Valid pipeline edge regions: {len(VALID_PIPELINE_EDGE_REGIONS):,}")
print(f"Valid road edge regions: {len(VALID_ROAD_EDGE_REGIONS):,}")

Valid node regions: 1,692
Valid pipeline edge regions: 5,886
Valid road edge regions: 1,518


In [23]:
# =============================================================================
# Replace baseline Region table with geospatial basemap regions
# =============================================================================

db_encoded = {
    table_name: df.copy()
    for table_name, df in db.items()
}

db_encoded["Region"] = region_table.copy()

print(f"Baseline Region rows: {len(db['Region']):,}")
print(f"Encoded Region rows: {len(db_encoded['Region']):,}")

display(
    db_encoded["Region"].head()
)

Baseline Region rows: 2,259
Encoded Region rows: 1,692


,region,notes
0,R0,1 degree CANOE geospatial graph node retained ...
1,R1,1 degree CANOE geospatial graph node retained ...
2,R2,1 degree CANOE geospatial graph node retained ...
3,R3,1 degree CANOE geospatial graph node retained ...
4,R4,1 degree CANOE geospatial graph node retained ...


In [24]:
# =============================================================================
# Define road transport technologies
# =============================================================================

truck_tech_specs = pd.DataFrame(
    [
        {
            "tech": "CO2_TRUCK",
            "input_comm": "co2",
            "output_comm": "co2",
            "description": "Road transport of carbon dioxide by truck",
        },
        {
            "tech": "H2_TRUCK",
            "input_comm": "h2",
            "output_comm": "h2",
            "description": "Road transport of hydrogen by truck",
        },
        {
            "tech": "GSL_TRUCK",
            "input_comm": "gsl",
            "output_comm": "gsl",
            "description": "Road transport of gasoline by truck",
        },
        {
            "tech": "METOH_TRUCK",
            "input_comm": "ch3oh",
            "output_comm": "ch3oh",
            "description": "Road transport of methanol by truck",
        },
    ]
)

display(truck_tech_specs)

,tech,input_comm,output_comm,description
0,CO2_TRUCK,co2,co2,Road transport of carbon dioxide by truck
1,H2_TRUCK,h2,h2,Road transport of hydrogen by truck
2,GSL_TRUCK,gsl,gsl,Road transport of gasoline by truck
3,METOH_TRUCK,ch3oh,ch3oh,Road transport of methanol by truck


In [25]:
# =============================================================================
# Define pipeline transport technologies
# =============================================================================

pipeline_tech_specs = pd.DataFrame(
    [
        {"tech": "CO2_PIPE", "input_comm": "co2", "output_comm": "co2"},
        {"tech": "H2_PIPE", "input_comm": "h2", "output_comm": "h2"},
        {"tech": "GSL_PIPE", "input_comm": "gsl", "output_comm": "gsl"},
        {"tech": "METOH_PIPE", "input_comm": "ch3oh", "output_comm": "ch3oh"},
    ]
)

pipeline_links = graph_edges.copy()
pipeline_links["canoe_region"] = pipeline_links["edge_region"]

assert len(pipeline_links) == len(VALID_PIPELINE_EDGE_REGIONS)
assert set(pipeline_links["canoe_region"]) == VALID_PIPELINE_EDGE_REGIONS

print(f"Pipeline links: {len(pipeline_links):,}")
display(pipeline_links.head())

Pipeline links: 5,886


,edge_region,region_from,region_to,direction,lon_from,lat_from,lon_to,lat_to,distance_km,resolution_deg,canoe_region
0,R0-R1,R0,R1,right,-81.5,43.5,-80.5,43.5,80.876192,1.0,R0-R1
1,R1-R3,R1,R3,up,-80.5,43.5,-80.5,44.5,111.112243,1.0,R1-R3
2,R1-R0,R1,R0,left,-80.5,43.5,-81.5,43.5,80.876192,1.0,R1-R0
3,R2-R8,R2,R8,up,-65.5,43.5,-65.5,44.5,111.112243,1.0,R2-R8
4,R3-R1,R3,R1,down,-80.5,44.5,-80.5,43.5,111.112243,1.0,R3-R1


In [26]:
# =============================================================================
# Build truck Technology rows
# =============================================================================

technology_template = (
    db["Technology"]
    .loc[
        db["Technology"]["tech"] == "H2_PIPE"
    ]
    .copy()
)

truck_technology = pd.concat(
    [
        technology_template.assign(
            tech=row.tech,
            description=row.description,
        )
        for row in truck_tech_specs.itertuples()
    ],
    ignore_index=True,
)

display(
    truck_technology
)

assert len(technology_template) == 1, (
    "Expected exactly one H2_PIPE technology row as template."
)

assert truck_technology["tech"].nunique() == len(truck_tech_specs), (
    "Truck technology rows are not unique."
)

,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
0,CO2_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of carbon dioxide by truck,GEO001
1,H2_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of hydrogen by truck,GEO001
2,GSL_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of gasoline by truck,GEO001
3,METOH_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of methanol by truck,GEO001


In [27]:
# =============================================================================
# Add truck technologies to encoded database
# =============================================================================

technology_template = (
    db_encoded["Technology"]
    .loc[db_encoded["Technology"]["tech"] == "H2_PIPE"]
    .copy()
)

assert len(technology_template) == 1, (
    "Expected exactly one H2_PIPE row to use as truck technology template."
)

truck_technology = pd.concat(
    [
        technology_template.assign(
            tech=row.tech,
            description=row.description,
        )
        for row in truck_tech_specs.itertuples(index=False)
    ],
    ignore_index=True,
)

db_encoded["Technology"] = (
    pd.concat(
        [
            db_encoded["Technology"],
            truck_technology,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["tech", "data_id"],
        keep="last",
    )
)

assert set(truck_tech_specs["tech"]).issubset(
    set(db_encoded["Technology"]["tech"])
)

print(f"Technology rows: {len(db_encoded['Technology']):,}")
display(db_encoded["Technology"].sort_values("tech"))

Technology rows: 16


,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
2,CO2_CAP,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
7,CO2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
12,CO2_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of carbon dioxide by truck,GEO001
0,ELC_GEN,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
5,ELC_TRANS,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
11,GSL_BACKUP,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
10,GSL_DEMAND,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
9,GSL_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
4,GSL_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
14,GSL_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of gasoline by truck,GEO001


In [28]:
# =============================================================================
# Build truck Efficiency rows for weak road links
# =============================================================================

assert (
    road_links_weak["canoe_region"].nunique()
    == len(road_links_weak)
), "Duplicate transport regions detected."

truck_efficiency_rows = []

for truck in truck_tech_specs.itertuples(index=False):

    df = pd.DataFrame(
        {
            "region": road_links_weak["canoe_region"],
            "input_comm": truck.input_comm,
            "tech": truck.tech,
            "vintage": 1,
            "output_comm": truck.output_comm,
            "efficiency": 1.0,
            "notes": "Existing weak road-connected transport link",
            "data_source": None,
            "dq_cred": None,
            "dq_geog": None,
            "dq_struc": None,
            "dq_tech": None,
            "dq_time": None,
            "data_id": "GEO001",
        }
    )

    truck_efficiency_rows.append(df)

truck_efficiency_weak = pd.concat(
    truck_efficiency_rows,
    ignore_index=True,
)

print(
    f"Truck Efficiency rows: "
    f"{len(truck_efficiency_weak):,}"
)

display(
    truck_efficiency_weak.head()
)

Truck Efficiency rows: 6,072


,region,input_comm,tech,vintage,output_comm,efficiency,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,R0-R1,co2,CO2_TRUCK,1,co2,1.0,Existing weak road-connected transport link,None,None,None,None,None,None,GEO001
1,R1-R3,co2,CO2_TRUCK,1,co2,1.0,Existing weak road-connected transport link,None,None,None,None,None,None,GEO001
2,R1-R0,co2,CO2_TRUCK,1,co2,1.0,Existing weak road-connected transport link,None,None,None,None,None,None,GEO001
3,R2-R8,co2,CO2_TRUCK,1,co2,1.0,Existing weak road-connected transport link,None,None,None,None,None,None,GEO001
4,R3-R1,co2,CO2_TRUCK,1,co2,1.0,Existing weak road-connected transport link,None,None,None,None,None,None,GEO001


In [29]:
# =============================================================================
# Build pipeline Efficiency rows for all graph links
# =============================================================================

assert (
    pipeline_links["canoe_region"].nunique()
    == len(pipeline_links)
), "Duplicate pipeline transport regions detected."

pipeline_efficiency_rows = []

for pipe in pipeline_tech_specs.itertuples(index=False):

    df = pd.DataFrame(
        {
            "region": pipeline_links["canoe_region"],
            "input_comm": pipe.input_comm,
            "tech": pipe.tech,
            "vintage": 1,
            "output_comm": pipe.output_comm,
            "efficiency": 1.0,
            "notes": "Candidate pipeline transport link on canonical graph edge",
            "data_source": None,
            "dq_cred": None,
            "dq_geog": None,
            "dq_struc": None,
            "dq_tech": None,
            "dq_time": None,
            "data_id": "GEO001",
        }
    )

    pipeline_efficiency_rows.append(df)

pipeline_efficiency = pd.concat(
    pipeline_efficiency_rows,
    ignore_index=True,
)

print(f"Pipeline Efficiency rows: {len(pipeline_efficiency):,}")
display(pipeline_efficiency.head())


Pipeline Efficiency rows: 23,544


,region,input_comm,tech,vintage,output_comm,efficiency,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,R0-R1,co2,CO2_PIPE,1,co2,1.0,Candidate pipeline transport link on canonical...,None,None,None,None,None,None,GEO001
1,R1-R3,co2,CO2_PIPE,1,co2,1.0,Candidate pipeline transport link on canonical...,None,None,None,None,None,None,GEO001
2,R1-R0,co2,CO2_PIPE,1,co2,1.0,Candidate pipeline transport link on canonical...,None,None,None,None,None,None,GEO001
3,R2-R8,co2,CO2_PIPE,1,co2,1.0,Candidate pipeline transport link on canonical...,None,None,None,None,None,None,GEO001
4,R3-R1,co2,CO2_PIPE,1,co2,1.0,Candidate pipeline transport link on canonical...,None,None,None,None,None,None,GEO001


In [30]:
# =============================================================================
# Add truck and pipeline Efficiency rows to database
# =============================================================================

db_encoded["Efficiency"] = (
    pd.concat(
        [
            db_encoded["Efficiency"],
            pipeline_efficiency,
            truck_efficiency_weak,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "region",
            "input_comm",
            "tech",
            "vintage",
            "output_comm",
            "data_id",
        ],
        keep="last",
    )
)

print(f"Encoded Efficiency rows: {len(db_encoded['Efficiency']):,}")

assert set(pipeline_efficiency["region"]) == VALID_PIPELINE_EDGE_REGIONS
assert set(truck_efficiency_weak["region"]) == VALID_ROAD_EDGE_REGIONS

Encoded Efficiency rows: 91,804


In [31]:
# =============================================================================
# Build placeholder truck CostVariable rows for weak road links
# =============================================================================

PLACEHOLDER_TRUCK_COST_PER_KM = 0.01
PLACEHOLDER_TRUCK_INTERCEPT_COST = 0.0

truck_costvariable_rows = []

assert (
    road_links_weak["distance_km"] > 0
).all(), "All transport distances must be positive."

assert (
    road_links_weak["canoe_region"].nunique()
    == len(road_links_weak)
), "Duplicate transport regions found."

for truck in truck_tech_specs.itertuples(index=False):
    df = pd.DataFrame(
        {
            "region": road_links_weak["canoe_region"],
            "period": 1,
            "tech": truck.tech,
            "vintage": 1,
            "cost": (
                PLACEHOLDER_TRUCK_INTERCEPT_COST
                + PLACEHOLDER_TRUCK_COST_PER_KM * road_links_weak["distance_km"]
            ),
            "units": "M$/unit",
            "notes": "Placeholder truck transport cost based on weak road-connected centroid distance",
            "data_source": None,
            "dq_cred": None,
            "dq_geog": None,
            "dq_struc": None,
            "dq_tech": None,
            "dq_time": None,
            "data_id": "GEO001",
        }
    )

    truck_costvariable_rows.append(df)

truck_costvariable_weak = pd.concat(
    truck_costvariable_rows,
    ignore_index=True,
)

print(f"Truck CostVariable rows: {len(truck_costvariable_weak):,}")

display(
    truck_costvariable_weak.head()
)

Truck CostVariable rows: 6,072


,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,R0-R1,1,CO2_TRUCK,1,0.808762,M$/unit,Placeholder truck transport cost based on weak...,None,None,None,None,None,None,GEO001
1,R1-R3,1,CO2_TRUCK,1,1.111122,M$/unit,Placeholder truck transport cost based on weak...,None,None,None,None,None,None,GEO001
2,R1-R0,1,CO2_TRUCK,1,0.808762,M$/unit,Placeholder truck transport cost based on weak...,None,None,None,None,None,None,GEO001
3,R2-R8,1,CO2_TRUCK,1,1.111122,M$/unit,Placeholder truck transport cost based on weak...,None,None,None,None,None,None,GEO001
4,R3-R1,1,CO2_TRUCK,1,1.111122,M$/unit,Placeholder truck transport cost based on weak...,None,None,None,None,None,None,GEO001


In [32]:
# =============================================================================
# Validate truck CostVariable rows
# =============================================================================

expected_costvariable_rows = len(road_links_weak) * len(truck_tech_specs)

assert len(truck_costvariable_weak) == expected_costvariable_rows

assert (
    truck_costvariable_weak[
        ["region", "period", "tech", "vintage", "data_id"]
    ].duplicated().sum() == 0
), "Duplicate CostVariable primary keys found."

assert (
    truck_costvariable_weak["cost"].notna().all()
), "Truck CostVariable contains missing costs."

assert (
    truck_costvariable_weak["cost"] >= 0
).all(), "Truck CostVariable contains negative costs."

print("Truck CostVariable rows validated.")

Truck CostVariable rows validated.


In [33]:
# =============================================================================
# Build placeholder pipeline CostVariable rows for graph links
# =============================================================================

PLACEHOLDER_PIPE_COST_PER_KM = 0.01
PLACEHOLDER_PIPE_INTERCEPT_COST = 0.0

pipeline_costvariable_rows = []

assert (
    pipeline_links["distance_km"] > 0
).all(), "All pipeline distances must be positive."

assert (
    pipeline_links["canoe_region"].nunique()
    == len(pipeline_links)
), "Duplicate pipeline transport regions found."

for pipe in pipeline_tech_specs.itertuples(index=False):

    df = pd.DataFrame(
        {
            "region": pipeline_links["canoe_region"],
            "period": 1,
            "tech": pipe.tech,
            "vintage": 1,
            "cost": (
                PLACEHOLDER_PIPE_INTERCEPT_COST
                + PLACEHOLDER_PIPE_COST_PER_KM
                * pipeline_links["distance_km"]
            ),
            "units": "M$/unit",
            "notes": "Placeholder pipeline transport cost based on graph-edge centroid distance",
            "data_source": None,
            "dq_cred": None,
            "dq_geog": None,
            "dq_struc": None,
            "dq_tech": None,
            "dq_time": None,
            "data_id": "GEO001",
        }
    )

    pipeline_costvariable_rows.append(df)

pipeline_costvariable = pd.concat(
    pipeline_costvariable_rows,
    ignore_index=True,
)

print(
    f"Pipeline CostVariable rows: "
    f"{len(pipeline_costvariable):,}"
)

display(
    pipeline_costvariable.head()
)

Pipeline CostVariable rows: 23,544


,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,R0-R1,1,CO2_PIPE,1,0.808762,M$/unit,Placeholder pipeline transport cost based on g...,None,None,None,None,None,None,GEO001
1,R1-R3,1,CO2_PIPE,1,1.111122,M$/unit,Placeholder pipeline transport cost based on g...,None,None,None,None,None,None,GEO001
2,R1-R0,1,CO2_PIPE,1,0.808762,M$/unit,Placeholder pipeline transport cost based on g...,None,None,None,None,None,None,GEO001
3,R2-R8,1,CO2_PIPE,1,1.111122,M$/unit,Placeholder pipeline transport cost based on g...,None,None,None,None,None,None,GEO001
4,R3-R1,1,CO2_PIPE,1,1.111122,M$/unit,Placeholder pipeline transport cost based on g...,None,None,None,None,None,None,GEO001


In [34]:
# =============================================================================
# Validate pipeline CostVariable rows
# =============================================================================

expected_pipeline_costvariable_rows = (
    len(pipeline_links) * len(pipeline_tech_specs)
)

assert len(pipeline_costvariable) == expected_pipeline_costvariable_rows

assert (
    pipeline_costvariable[
        ["region", "period", "tech", "vintage", "data_id"]
    ].duplicated().sum() == 0
), "Duplicate pipeline CostVariable primary keys found."

assert pipeline_costvariable["cost"].notna().all()

assert (pipeline_costvariable["cost"] >= 0).all()

assert set(pipeline_costvariable["region"]) == VALID_PIPELINE_EDGE_REGIONS

print("Pipeline CostVariable rows validated.")

Pipeline CostVariable rows validated.


In [35]:
# =============================================================================
# Add transport CostVariable rows to database
# =============================================================================

db_encoded["CostVariable"] = (
    pd.concat(
        [
            db_encoded["CostVariable"],
            pipeline_costvariable,
            truck_costvariable_weak,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "region",
            "period",
            "tech",
            "vintage",
            "data_id",
        ],
        keep="last",
    )
)

pipeline_regions = set(
    pipeline_costvariable["region"]
)

truck_regions = set(
    truck_costvariable_weak["region"]
)

assert pipeline_regions == VALID_PIPELINE_EDGE_REGIONS
assert truck_regions == VALID_ROAD_EDGE_REGIONS

print(
    f"Pipeline CostVariable regions: "
    f"{len(pipeline_regions):,}"
)

print(
    f"Truck CostVariable regions: "
    f"{len(truck_regions):,}"
)

print(
    f"Encoded CostVariable rows: "
    f"{len(db_encoded['CostVariable']):,}"
)

Pipeline CostVariable regions: 5,886
Truck CostVariable regions: 1,518
Encoded CostVariable rows: 68,159


In [36]:
# =============================================================================
# Define graph region sets by transport layer
# =============================================================================

VALID_NODE_REGIONS = set(
    region_table["region"]
)

VALID_PIPELINE_EDGE_REGIONS = set(
    graph_edges["edge_region"]
)

VALID_ROAD_EDGE_REGIONS = set(
    road_links_weak["canoe_region"]
)

VALID_ALL_EDGE_REGIONS = (
    VALID_PIPELINE_EDGE_REGIONS
    | VALID_ROAD_EDGE_REGIONS
)

print(
    f"Valid node regions: "
    f"{len(VALID_NODE_REGIONS):,}"
)

print(
    f"Valid pipeline edge regions: "
    f"{len(VALID_PIPELINE_EDGE_REGIONS):,}"
)

print(
    f"Valid road edge regions: "
    f"{len(VALID_ROAD_EDGE_REGIONS):,}"
)

Valid node regions: 1,692
Valid pipeline edge regions: 5,886
Valid road edge regions: 1,518


In [37]:
# =============================================================================
# Helper: filter rows by valid graph regions
# =============================================================================

def filter_valid_regions(
    df: pd.DataFrame,
    valid_node_regions: set[str],
    valid_edge_regions: set[str],
    region_col: str = "region",
) -> pd.DataFrame:
    """
    Keep rows whose region is either

    1. a valid node region, or
    2. a valid edge region for the transport layer being considered.
    """

    if region_col not in df.columns:
        return df.copy()

    keep_mask = (
        df[region_col].isin(valid_node_regions)
        |
        df[region_col].isin(valid_edge_regions)
    )

    return (
        df.loc[keep_mask]
        .copy()
        .reset_index(drop=True)
    )

In [38]:
# =============================================================================
# Check node-region references after Region table replacement
# =============================================================================

region_reference_report = []

for table_name, df in db_encoded.items():
    if "region" not in df.columns:
        continue

    region_values = df["region"].dropna().astype(str)

    node_region_values = region_values[
        ~region_values.str.contains("-", regex=False)
    ]

    invalid_node_regions = sorted(
        set(node_region_values) - VALID_NODE_REGIONS
    )

    region_reference_report.append(
        {
            "table": table_name,
            "rows": len(df),
            "node_region_values": node_region_values.nunique(),
            "invalid_node_regions": len(invalid_node_regions),
            "example_invalid_regions": invalid_node_regions[:10],
        }
    )

region_reference_report = pd.DataFrame(region_reference_report)

display(
    region_reference_report.sort_values(
        "invalid_node_regions",
        ascending=False,
    )
)

,table,rows,node_region_values,invalid_node_regions,example_invalid_regions
8,CostVariable,68159,2259,567,"[R1692, R1693, R1694, R1695, R1696, R1697, R16..."
7,CostInvest,4518,2259,567,"[R1692, R1693, R1694, R1695, R1696, R1697, R16..."
12,Efficiency,91804,2259,567,"[R1692, R1693, R1694, R1695, R1696, R1697, R16..."
17,ETLSegment,193552,2259,567,"[R1692, R1693, R1694, R1695, R1696, R1697, R16..."
33,LimitCapacity,4518,2259,567,"[R1692, R1693, R1694, R1695, R1696, R1697, R16..."
...,...,...,...,...,...
49,StorageDuration,0,0,0,[]
48,ReserveCapacityDerate,0,0,0,[]
55,OutputRetiredCapacity,0,0,0,[]
58,OutputStorageLevel,0,0,0,[]


In [39]:
# =============================================================================
# Filter edge-region tables to valid graph regions
#
# Remove inherited transport links whose regions are not present in the
# canonical graph. Retain only node regions contained in VALID_NODE_REGIONS
# and edge pseudo-regions contained in VALID_EDGE_REGIONS.
# =============================================================================

edge_region_tables = [
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
]

for table_name in edge_region_tables:

    before_rows = len(
        db_encoded[table_name]
    )

    db_encoded[table_name] = filter_valid_regions(
        db_encoded[table_name],
        VALID_NODE_REGIONS,
        valid_edge_regions=VALID_PIPELINE_EDGE_REGIONS,
    )

    after_rows = len(
        db_encoded[table_name]
    )

    print(
        f"{table_name}: "
        f"{before_rows:,} → "
        f"{after_rows:,} "
        f"({after_rows - before_rows:+,})"
    )

Efficiency: 91,804 → 58,254 (-33,550)
CostVariable: 68,159 → 37,678 (-30,481)
CostInvest: 4,518 → 3,384 (-1,134)
ETLSegment: 193,552 → 73,256 (-120,296)


In [40]:
# =============================================================================
# Build zero CostInvest rows for truck road links
# =============================================================================

truck_costinvest_rows = []

assert set(road_links_weak["canoe_region"]).issubset(VALID_ROAD_EDGE_REGIONS), (
    "Truck CostInvest regions must be valid road graph edges."
)

for truck in truck_tech_specs.itertuples(index=False):
    df = pd.DataFrame(
        {
            "region": road_links_weak["canoe_region"],
            "tech": truck.tech,
            "vintage": 1,
            "cost": 0.0,
            "units": "M$/unit",
            "notes": "Existing road transport link; no road construction investment encoded",
            "data_source": None,
            "dq_cred": None,
            "dq_geog": None,
            "dq_struc": None,
            "dq_tech": None,
            "dq_time": None,
            "data_id": "GEO001",
        }
    )

    truck_costinvest_rows.append(df)

truck_costinvest_weak = pd.concat(
    truck_costinvest_rows,
    ignore_index=True,
)

print(f"Truck CostInvest rows: {len(truck_costinvest_weak):,}")

display(
    truck_costinvest_weak.head()
)

Truck CostInvest rows: 6,072


,region,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,R0-R1,CO2_TRUCK,1,0.0,M$/unit,Existing road transport link; no road construc...,None,None,None,None,None,None,GEO001
1,R1-R3,CO2_TRUCK,1,0.0,M$/unit,Existing road transport link; no road construc...,None,None,None,None,None,None,GEO001
2,R1-R0,CO2_TRUCK,1,0.0,M$/unit,Existing road transport link; no road construc...,None,None,None,None,None,None,GEO001
3,R2-R8,CO2_TRUCK,1,0.0,M$/unit,Existing road transport link; no road construc...,None,None,None,None,None,None,GEO001
4,R3-R1,CO2_TRUCK,1,0.0,M$/unit,Existing road transport link; no road construc...,None,None,None,None,None,None,GEO001


In [41]:
# =============================================================================
# Validate truck CostInvest rows
# =============================================================================

expected_costinvest_rows = len(road_links_weak) * len(truck_tech_specs)

assert len(truck_costinvest_weak) == expected_costinvest_rows

assert (
    truck_costinvest_weak[
        ["region", "tech", "vintage", "data_id"]
    ].duplicated().sum() == 0
), "Duplicate CostInvest primary keys found."

assert (
    truck_costinvest_weak["cost"] == 0
).all(), "Truck CostInvest rows should have zero investment cost."

print("Truck CostInvest rows validated.")

Truck CostInvest rows validated.


In [42]:
# =============================================================================
# Add truck CostInvest rows to database
# =============================================================================

db_encoded["CostInvest"] = (
    pd.concat(
        [
            db_encoded["CostInvest"],
            truck_costinvest_weak,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "region",
            "tech",
            "vintage",
            "data_id",
        ],
        keep="last",
    )
)

print(f"Encoded CostInvest rows: {len(db_encoded['CostInvest']):,}")

Encoded CostInvest rows: 9,456


In [43]:
# =============================================================================
# Summarize encoded database changes
# =============================================================================

summary_rows = []

for table in [
    "Region",
    "Technology",
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
]:
    summary_rows.append(
        {
            "table": table,
            "baseline_rows": len(db[table]),
            "encoded_rows": len(db_encoded[table]),
            "delta_rows": (
                len(db_encoded[table])
                - len(db[table])
            ),
        }
    )

encoding_summary = pd.DataFrame(
    summary_rows
)

display(
    encoding_summary
)

,table,baseline_rows,encoded_rows,delta_rows
0,Region,2259,1692,-567
1,Technology,12,16,4
2,Efficiency,62188,58254,-3934
3,CostVariable,50487,37678,-12809
4,CostInvest,4518,9456,4938
5,ETLSegment,193552,73256,-120296


In [44]:
# =============================================================================
# Summarize transport edge regions
# =============================================================================

for table in [
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
]:

    edge_regions = (
        db_encoded[table]["region"]
        .dropna()
        .astype(str)
    )

    edge_regions = edge_regions[
        edge_regions.str.contains(
            "-",
            regex=False,
        )
    ]

    print(
        f"{table}: "
        f"{edge_regions.nunique():,} "
        f"transport edge regions"
    )

Efficiency: 5,886 transport edge regions
CostVariable: 5,886 transport edge regions
CostInvest: 1,518 transport edge regions
ETLSegment: 2,986 transport edge regions


## Reconcile inherited baseline tables with geospatial regions

The baseline SQLite database contains a complete runnable CANOE test model, but several inherited tables were generated using the old synthetic site aggregation workflow. Since the Region table has now been replaced by the geospatial 1° basemap regions, inherited node-based tables may contain references to regions that no longer exist.

The next cells identify and filter or rebuild these tables so that all node-region references are consistent with the geospatial Region table.

In [45]:
# =============================================================================
# Helper: filter tables to valid canonical graph regions
# =============================================================================

def filter_valid_graph_regions(
    df: pd.DataFrame,
    VALID_NODE_REGIONS: set[str],
    valid_edge_regions: set[str],
    region_col: str = "region",
) -> pd.DataFrame:
    """
    Keep rows whose region is either:
    1. a valid graph node region, or
    2. a valid canonical graph edge region.
    """

    if region_col not in df.columns:
        return df.copy()

    region_values = df[region_col].astype(str)

    keep_mask = (
        region_values.isin(VALID_NODE_REGIONS)
        |
        region_values.isin(valid_edge_regions)
    )

    return (
        df.loc[keep_mask]
        .copy()
        .reset_index(drop=True)
    )

In [46]:
# =============================================================================
# Load old model site dictionary for comparison
# =============================================================================

OLD_SITE_DICT_PATH = PROJECT_ROOT / "sites_dict_1.csv"

old_sites = pd.read_csv(OLD_SITE_DICT_PATH)

print(f"Old site dictionary rows: {len(old_sites):,}")
print(f"New graph node regions: {len(graph_nodes):,}")

display(
    old_sites.head()
)

display(
    old_sites.columns
)

Old site dictionary rows: 2,259
New graph node regions: 1,692


,lon,lat,LCOE,max_elc,demand,co2,region,up_id,down_id,right_id,left_id,right_distance,left_distance,up_distance,down_distance,site_id,co2_cost
0,-141,60,0.056970,9.904231e+05,0.0,0.0,R0,R1,R-999,R11,R-999,55.799470,NaN,111.420728,NaN,R0,50
1,-141,61,0.100296,1.805530e+06,0.0,0.0,R1,R2,R0,R12,R-999,54.106953,NaN,111.437373,111.420728,R1,50
2,-141,62,0.138735,9.350884e+05,0.0,0.0,R2,R3,R1,R13,R-999,52.397727,NaN,111.453649,111.437373,R2,50
3,-141,63,0.120369,1.248552e+06,0.0,0.0,R3,R4,R2,R14,R-999,50.672313,NaN,111.469535,111.453649,R3,50
4,-141,64,0.080409,1.285275e+06,0.0,0.0,R4,R5,R3,R15,R-999,48.931240,NaN,111.485013,111.469535,R4,50


Index(['lon', 'lat', 'LCOE', 'max_elc', 'demand', 'co2', 'region', 'up_id',
       'down_id', 'right_id', 'left_id', 'right_distance', 'left_distance',
       'up_distance', 'down_distance', 'site_id', 'co2_cost'],
      dtype='object')

In [47]:
# =============================================================================
# Compare old and new region ID coverage
# =============================================================================

old_site_regions = set(old_sites["region"])
new_region_ids = set(region_table["region"])

shared_regions = old_site_regions & new_region_ids
old_only_regions = old_site_regions - new_region_ids
new_only_regions = new_region_ids - old_site_regions

print(f"Shared regions: {len(shared_regions):,}")
print(f"Old-only regions: {len(old_only_regions):,}")
print(f"New-only regions: {len(new_only_regions):,}")

print("\nExample old-only regions:")
print(sorted(old_only_regions, key=lambda x: int(x.replace("R", "")))[:20])

print("\nExample new-only regions:")
print(sorted(new_only_regions, key=lambda x: int(x.replace("R", "")))[:20])

Shared regions: 1,692
Old-only regions: 567
New-only regions: 0

Example old-only regions:
['R1692', 'R1693', 'R1694', 'R1695', 'R1696', 'R1697', 'R1698', 'R1699', 'R1700', 'R1701', 'R1702', 'R1703', 'R1704', 'R1705', 'R1706', 'R1707', 'R1708', 'R1709', 'R1710', 'R1711']

Example new-only regions:
[]


In [48]:
# =============================================================================
# Filter inherited region tables to valid graph regions
# =============================================================================

tables_to_filter_by_graph_region = [
    "Demand",
    "LimitCapacity",
    "LimitTechInputSplitAnnual",
    "CostInvest",
    "CostVariable",
    "Efficiency",
]

for table_name in tables_to_filter_by_graph_region:

    before_rows = len(
        db_encoded[table_name]
    )

    db_encoded[table_name] = filter_valid_regions(
        db_encoded[table_name],
        valid_node_regions=VALID_NODE_REGIONS,
        valid_edge_regions=VALID_PIPELINE_EDGE_REGIONS,
    )

    after_rows = len(
        db_encoded[table_name]
    )

    print(
        f"{table_name}: "
        f"{before_rows:,} → "
        f"{after_rows:,} "
        f"({after_rows - before_rows:+,})"
    )

Demand: 123 → 86 (-37)
LimitCapacity: 4,518 → 3,384 (-1,134)
LimitTechInputSplitAnnual: 11,295 → 8,460 (-2,835)
CostInvest: 9,456 → 9,456 (+0)
CostVariable: 37,678 → 37,678 (+0)
Efficiency: 58,254 → 58,254 (+0)


In [49]:
# =============================================================================
# Re-check edge-region references after filtering
# =============================================================================

for table_name in [
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
]:

    edge_values = (
        db_encoded[table_name]["region"]
        .dropna()
        .astype(str)
    )

    edge_values = edge_values[
        edge_values.str.contains("-", regex=False)
    ]

    invalid_edge_regions = sorted(
    set(edge_values) - VALID_PIPELINE_EDGE_REGIONS
    )

    print(
        f"{table_name}: "
        f"{edge_values.nunique():,} edge regions, "
        f"{len(invalid_edge_regions):,} invalid"
    )

    if invalid_edge_regions:
        print(invalid_edge_regions[:10])

Efficiency: 5,886 edge regions, 0 invalid
CostVariable: 5,886 edge regions, 0 invalid
CostInvest: 1,518 edge regions, 0 invalid
ETLSegment: 2,986 edge regions, 0 invalid


In [50]:
# =============================================================================
# Inspect ETLSegment technologies
# =============================================================================

etl_tech_counts = (
    db_encoded["ETLSegment"]
    .groupby("tech_or_group")
    .size()
    .reset_index(name="rows")
    .sort_values(
        "rows",
        ascending=False,
    )
)

display(
    etl_tech_counts
)

,tech_or_group,rows
0,CO2_PIPE,11944
1,ELC_TRANS,11944
2,GSL_PIPE,11944
4,H2_PIPE,11944
5,METOH_PIPE,11944
3,GSL_PLANT,6768
6,METOH_PLANT,6768


In [51]:
# =============================================================================
# Inspect pipeline ETLSegment rows
# =============================================================================

pipeline_etl = db_encoded["ETLSegment"].loc[
    db_encoded["ETLSegment"]["tech_or_group"].isin(
        [
            "H2_PIPE",
            "CO2_PIPE",
            "GSL_PIPE",
            "METOH_PIPE",
        ]
    )
]

print(
    f"Pipeline ETLSegment rows: {len(pipeline_etl):,}"
)

display(
    pipeline_etl.head()
)

Pipeline ETLSegment rows: 47,776


,region,tech_or_group,segment,cap_lower,cap_upper,cost_lower,cost_upper,data_id
13536,R0-R1,GSL_PIPE,0,0.0,86475.490004,0.0,1.836283e+08,GEO001
13537,R1-R0,GSL_PIPE,0,0.0,86475.490004,0.0,1.836283e+08,GEO001
13538,R3-R4,GSL_PIPE,0,0.0,86475.490004,0.0,1.836283e+08,GEO001
13539,R4-R5,GSL_PIPE,0,0.0,86475.490004,0.0,1.836283e+08,GEO001
13540,R4-R3,GSL_PIPE,0,0.0,86475.490004,0.0,1.836283e+08,GEO001


In [52]:
# =============================================================================
# Unique pipeline transport regions
# =============================================================================

pipeline_edge_regions = (
    pipeline_etl["region"]
    .dropna()
    .astype(str)
)

pipeline_edge_regions = pipeline_edge_regions[
    pipeline_edge_regions.str.contains("-", regex=False)
]

print(
    f"Unique pipeline edge regions: "
    f"{pipeline_edge_regions.nunique():,}"
)

print(
    f"Canonical graph edges: "
    f"{len(VALID_PIPELINE_EDGE_REGIONS):,}"
)

pipe_eff_regions = set(
    db_encoded["Efficiency"]
    .loc[
        db_encoded["Efficiency"]["tech"].isin(
            ["H2_PIPE", "CO2_PIPE", "GSL_PIPE", "METOH_PIPE"]
        ),
        "region",
    ]
    .astype(str)
)

pipe_etl_regions = set(
    pipeline_etl["region"].astype(str)
)

missing_pipe_etl = sorted(
    pipe_eff_regions - pipe_etl_regions
)

print(f"Pipeline Efficiency edge regions: {len(pipe_eff_regions):,}")
print(f"Pipeline ETLSegment edge regions: {len(pipe_etl_regions):,}")
print(f"Pipeline Efficiency regions missing ETLSegment: {len(missing_pipe_etl):,}")
print(missing_pipe_etl[:20])

Unique pipeline edge regions: 2,986
Canonical graph edges: 5,886
Pipeline Efficiency edge regions: 5,886
Pipeline ETLSegment edge regions: 2,986
Pipeline Efficiency regions missing ETLSegment: 2,900
['R1-R3', 'R10-R11', 'R10-R30', 'R10-R4', 'R100-R163', 'R100-R66', 'R1000-R1058', 'R1000-R946', 'R1001-R1059', 'R1001-R947', 'R1002-R1060', 'R1002-R948', 'R1003-R1061', 'R1003-R949', 'R1004-R1062', 'R1004-R950', 'R1005-R1063', 'R1005-R951', 'R1006-R1064', 'R1007-R1065']


In [53]:
# =============================================================================
# Characterize ETLSegment region types
# =============================================================================

etl = db_encoded["ETLSegment"].copy()

etl["is_edge_region"] = (
    etl["region"]
    .astype(str)
    .str.contains("-", regex=False)
)

etl_region_summary = (
    etl
    .groupby(
        ["tech_or_group", "is_edge_region"]
    )
    .size()
    .reset_index(name="rows")
    .sort_values(
        ["tech_or_group", "is_edge_region"]
    )
)

display(etl_region_summary)

,tech_or_group,is_edge_region,rows
0,CO2_PIPE,True,11944
1,ELC_TRANS,True,11944
2,GSL_PIPE,True,11944
3,GSL_PLANT,False,6768
4,H2_PIPE,True,11944
5,METOH_PIPE,True,11944
6,METOH_PLANT,False,6768


In [54]:
# =============================================================================
# Characterize Efficiency region types
# =============================================================================

eff = db_encoded["Efficiency"].copy()

eff["is_edge_region"] = (
    eff["region"]
    .astype(str)
    .str.contains("-", regex=False)
)

eff_region_summary = (
    eff
    .groupby(
        ["tech", "is_edge_region"]
    )
    .size()
    .reset_index(name="rows")
    .sort_values(
        ["tech", "is_edge_region"]
    )
)

display(eff_region_summary)

,tech,is_edge_region,rows
0,CO2_CAP,False,1692
1,CO2_PIPE,True,8872
2,CO2_TRUCK,True,1518
3,ELC_GEN,False,1692
4,ELC_TRANS,True,2986
5,GSL_BACKUP,False,86
6,GSL_DEMAND,False,86
7,GSL_PIPE,True,8872
8,GSL_PLANT,False,3384
9,GSL_TRUCK,True,1518


In [55]:
# =============================================================================
# Check ETLSegment node-region validity
# =============================================================================

etl = db_encoded["ETLSegment"].copy()

etl["is_edge_region"] = (
    etl["region"]
    .astype(str)
    .str.contains("-", regex=False)
)

node_etl = etl.loc[
    ~etl["is_edge_region"]
].copy()

invalid_node_etl = (
    ~node_etl["region"].isin(VALID_NODE_REGIONS)
)

print(f"Node ETLSegment rows: {len(node_etl):,}")
print(f"Invalid node ETLSegment rows: {invalid_node_etl.sum():,}")
print(f"Valid node ETLSegment rows: {(~invalid_node_etl).sum():,}")

display(
    node_etl.loc[
        invalid_node_etl
    ].head()
)

Node ETLSegment rows: 13,536
Invalid node ETLSegment rows: 0
Valid node ETLSegment rows: 13,536


,region,tech_or_group,segment,cap_lower,cap_upper,cost_lower,cost_upper,data_id,is_edge_region


In [56]:
# =============================================================================
# Filter ETLSegment to valid graph regions
# =============================================================================

before_rows = len(
    db_encoded["ETLSegment"]
)

db_encoded["ETLSegment"] = filter_valid_regions(
    db_encoded["ETLSegment"],
    valid_node_regions=VALID_NODE_REGIONS,
    valid_edge_regions=VALID_PIPELINE_EDGE_REGIONS,
)

after_rows = len(
    db_encoded["ETLSegment"]
)

print(
    f"ETLSegment: "
    f"{before_rows:,} → "
    f"{after_rows:,} "
    f"({after_rows - before_rows:+,})"
)

ETLSegment: 73,256 → 73,256 (+0)


In [57]:
# =============================================================================
# Build pipeline ETLSegment rows for all graph links
# =============================================================================

pipeline_etl_template = (
    db_encoded["ETLSegment"]
    .loc[
        db_encoded["ETLSegment"]["tech_or_group"] == "H2_PIPE"
    ]
    .head(1)
    .copy()
)

assert len(pipeline_etl_template) == 1, (
    "Expected one H2_PIPE ETLSegment row as template."
)

pipeline_etl_rows = []

for pipe in pipeline_tech_specs.itertuples(index=False):

    df = pd.DataFrame(
        np.repeat(
            pipeline_etl_template.values,
            len(pipeline_links),
            axis=0,
        ),
        columns=pipeline_etl_template.columns,
    )

    df["region"] = pipeline_links["canoe_region"].values
    df["tech_or_group"] = pipe.tech

    pipeline_etl_rows.append(df)

pipeline_etl_new = pd.concat(
    pipeline_etl_rows,
    ignore_index=True,
)

print(
    f"Pipeline ETLSegment rows: "
    f"{len(pipeline_etl_new):,}"
)

display(
    pipeline_etl_new.head()
)

Pipeline ETLSegment rows: 23,544


,region,tech_or_group,segment,cap_lower,cap_upper,cost_lower,cost_upper,data_id
0,R0-R1,CO2_PIPE,0,0.0,86475.490004,0.0,183628292.496408,GEO001
1,R1-R3,CO2_PIPE,0,0.0,86475.490004,0.0,183628292.496408,GEO001
2,R1-R0,CO2_PIPE,0,0.0,86475.490004,0.0,183628292.496408,GEO001
3,R2-R8,CO2_PIPE,0,0.0,86475.490004,0.0,183628292.496408,GEO001
4,R3-R1,CO2_PIPE,0,0.0,86475.490004,0.0,183628292.496408,GEO001


In [58]:
# =============================================================================
# Replace pipeline ETLSegment rows in database
# =============================================================================

PIPE_TECHS = set(pipeline_tech_specs["tech"])

before_rows = len(db_encoded["ETLSegment"])

non_pipeline_etl = db_encoded["ETLSegment"].loc[
    ~db_encoded["ETLSegment"]["tech_or_group"].isin(PIPE_TECHS)
].copy()

db_encoded["ETLSegment"] = pd.concat(
    [
        non_pipeline_etl,
        pipeline_etl_new,
    ],
    ignore_index=True,
)

after_rows = len(db_encoded["ETLSegment"])

print(
    f"ETLSegment: {before_rows:,} → {after_rows:,} "
    f"({after_rows - before_rows:+,})"
)

print(
    f"Pipeline ETLSegment regions: "
    f"{pipeline_etl_new['region'].nunique():,}"
)

ETLSegment: 73,256 → 49,024 (-24,232)
Pipeline ETLSegment regions: 5,886


In [59]:
# Pipeline ETLSegment coverage check
pipeline_etl = db_encoded["ETLSegment"].loc[
    db_encoded["ETLSegment"]["tech_or_group"].isin(pipeline_tech_specs["tech"])
]

pipeline_etl_regions = set(pipeline_etl["region"].astype(str))
pipeline_eff_regions = set(
    db_encoded["Efficiency"]
    .loc[
        db_encoded["Efficiency"]["tech"].isin(pipeline_tech_specs["tech"]),
        "region",
    ]
    .astype(str)
)

print(f"Pipeline Efficiency regions: {len(pipeline_eff_regions):,}")
print(f"Pipeline ETLSegment regions: {len(pipeline_etl_regions):,}")
print(f"Missing ETLSegment regions: {len(pipeline_eff_regions - pipeline_etl_regions):,}")
print(f"Extra ETLSegment regions: {len(pipeline_etl_regions - pipeline_eff_regions):,}")

Pipeline Efficiency regions: 5,886
Pipeline ETLSegment regions: 5,886
Missing ETLSegment regions: 0
Extra ETLSegment regions: 0


In [60]:
# =============================================================================
# Add pipeline ETLSegment rows to database
# =============================================================================

db_encoded["ETLSegment"] = (
    pd.concat(
        [
            db_encoded["ETLSegment"],
            pipeline_etl_new,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "region",
            "tech_or_group",
            "segment",
            "data_id",
        ],
        keep="last",
    )
)

print(
    f"Encoded ETLSegment rows: "
    f"{len(db_encoded['ETLSegment']):,}"
)

Encoded ETLSegment rows: 49,024


In [61]:
# =============================================================================
# Final node-region consistency check
# =============================================================================

region_reference_report_final = []

for table_name, df in db_encoded.items():

    if "region" not in df.columns:
        continue

    region_values = df["region"].dropna().astype(str)

    node_region_values = region_values[
        ~region_values.str.contains("-", regex=False)
    ]

    invalid_node_regions = sorted(
        set(node_region_values) - VALID_NODE_REGIONS
    )

    region_reference_report_final.append(
        {
            "table": table_name,
            "rows": len(df),
            "node_region_values": node_region_values.nunique(),
            "invalid_node_regions": len(invalid_node_regions),
            "example_invalid_regions": invalid_node_regions[:10],
        }
    )

region_reference_report_final = pd.DataFrame(
    region_reference_report_final
)

display(
    region_reference_report_final.sort_values(
        "invalid_node_regions",
        ascending=False,
    )
)

,table,rows,node_region_values,invalid_node_regions,example_invalid_regions
60,OutputCost,2187,613,165,"[R1716, R1719, R1720, R1721, R1722, R1723, R17..."
56,OutputFlowIn,1608,155,47,"[R1716, R1719, R1725, R1756, R1766, R1787, R17..."
57,OutputFlowOut,1608,155,47,"[R1716, R1719, R1725, R1756, R1766, R1787, R17..."
53,OutputNetCapacity,1693,154,46,"[R1716, R1719, R1725, R1756, R1766, R1787, R17..."
54,OutputBuiltCapacity,1693,154,46,"[R1716, R1719, R1725, R1756, R1766, R1787, R17..."
...,...,...,...,...,...
49,StorageDuration,0,0,0,[]
48,ReserveCapacityDerate,0,0,0,[]
55,OutputRetiredCapacity,0,0,0,[]
58,OutputStorageLevel,0,0,0,[]


In [62]:
# =============================================================================
# Final edge-region consistency check
# =============================================================================

for table_name in [
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
]:

    edge_values = (
        db_encoded[table_name]["region"]
        .dropna()
        .astype(str)
    )

    edge_values = edge_values[
        edge_values.str.contains("-", regex=False)
    ]

    invalid_edge_regions = sorted(
    set(edge_values) - VALID_ALL_EDGE_REGIONS
    )

    print(
        f"{table_name}: "
        f"{edge_values.nunique():,} edge regions, "
        f"{len(invalid_edge_regions):,} invalid"
    )

Efficiency: 5,886 edge regions, 0 invalid
CostVariable: 5,886 edge regions, 0 invalid
CostInvest: 1,518 edge regions, 0 invalid
ETLSegment: 5,886 edge regions, 0 invalid


In [63]:
# =============================================================================
# Clear solver output tables
# =============================================================================

output_tables = [
    table_name
    for table_name in db_encoded
    if table_name.startswith("Output")
]

for table_name in output_tables:

    before_rows = len(db_encoded[table_name])

    db_encoded[table_name] = (
        db_encoded[table_name]
        .iloc[0:0]
        .copy()
    )

    print(
        f"{table_name}: {before_rows:,} → {len(db_encoded[table_name]):,}"
    )

OutputCurtailment: 0 → 0
OutputNetCapacity: 1,693 → 0
OutputBuiltCapacity: 1,693 → 0
OutputRetiredCapacity: 0 → 0
OutputFlowIn: 1,608 → 0
OutputFlowOut: 1,608 → 0
OutputStorageLevel: 0 → 0
OutputDualVariable: 0 → 0
OutputObjective: 1 → 0
OutputEmission: 0 → 0
OutputCost: 2,187 → 0


In [64]:
# =============================================================================
# Re-run final node-region consistency check after clearing outputs
# =============================================================================

region_reference_report_final = []

for table_name, df in db_encoded.items():

    if "region" not in df.columns:
        continue

    region_values = df["region"].dropna().astype(str)

    node_region_values = region_values[
        ~region_values.str.contains("-", regex=False)
    ]

    invalid_node_regions = sorted(
        set(node_region_values) - VALID_NODE_REGIONS
    )

    region_reference_report_final.append(
        {
            "table": table_name,
            "rows": len(df),
            "node_region_values": node_region_values.nunique(),
            "invalid_node_regions": len(invalid_node_regions),
            "example_invalid_regions": invalid_node_regions[:10],
        }
    )

region_reference_report_final = pd.DataFrame(
    region_reference_report_final
)

display(
    region_reference_report_final.sort_values(
        "invalid_node_regions",
        ascending=False,
    )
)

assert (
    region_reference_report_final["invalid_node_regions"] == 0
).all(), "Invalid node-region references remain."

print("All node-region references are valid.")

,table,rows,node_region_values,invalid_node_regions,example_invalid_regions
0,CapacityCredit,0,0,0,[]
1,CapacityFactorProcess,0,0,0,[]
2,CapacityFactorTech,0,0,0,[]
3,CapacityToActivity,0,0,0,[]
4,ConstructionInput,0,0,0,[]
...,...,...,...,...,...
56,OutputFlowIn,0,0,0,[]
57,OutputFlowOut,0,0,0,[]
58,OutputStorageLevel,0,0,0,[]
59,OutputEmission,0,0,0,[]


All node-region references are valid.


In [65]:
# =============================================================================
# Final encoded database summary
# =============================================================================

final_summary = []

for table_name, df in db_encoded.items():

    final_summary.append(
        {
            "table": table_name,
            "rows": len(df),
            "columns": len(df.columns),
        }
    )

final_summary = (
    pd.DataFrame(final_summary)
    .sort_values(
        "table"
    )
    .reset_index(drop=True)
)

display(final_summary)

,table,rows,columns
0,CapacityCredit,0,13
1,CapacityFactorProcess,0,15
2,CapacityFactorTech,0,14
3,CapacityToActivity,0,11
4,Commodity,7,4
...,...,...,...
80,Technology,16,15
81,TechnologyType,4,2
82,TimePeriod,2,3
83,TimePeriodType,2,2


In [66]:
# =============================================================================
# Create fresh weak-road SQLite database
# =============================================================================

if OUTPUT_SQLITE_WEAK_PATH.exists():
    OUTPUT_SQLITE_WEAK_PATH.unlink()

db_mgmt.convert_sql_to_sqlite(
    RAW_SCHEMA_PATH,
    OUTPUT_SQLITE_WEAK_PATH,
)

print(
    f"Created {OUTPUT_SQLITE_WEAK_PATH.name}"
)

Created CANOE_geospatial_1deg_graph_roads_weak.sqlite


In [67]:
# =============================================================================
# Write encoded tables to SQLite
# =============================================================================

db_mgmt.update_sqlite(
    OUTPUT_SQLITE_WEAK_PATH,
    db_encoded,
)

print(
    f"Wrote {len(db_encoded)} tables to {OUTPUT_SQLITE_WEAK_PATH.name}"
)

Wrote 85 tables to CANOE_geospatial_1deg_graph_roads_weak.sqlite


In [68]:
# =============================================================================
# Verify exported SQLite database
# =============================================================================

db_test = db_mgmt.sqlite_to_dfs(
    OUTPUT_SQLITE_WEAK_PATH
)

print(
    f"Exported tables: {len(db_test)}"
)

for table_name in [
    "Region",
    "Technology",
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
]:
    print(
        f"{table_name}: {len(db_test[table_name]):,}"
    )

Exported tables: 85
Region: 1,692
Technology: 16
Efficiency: 58,254
CostVariable: 37,678
CostInvest: 9,456
ETLSegment: 49,024


In [69]:
# =============================================================================
# Check truck technologies in working and exported databases
# =============================================================================

truck_tech_names = set(truck_tech_specs["tech"])

print("In db_encoded:")
print(
    sorted(
        truck_tech_names
        & set(db_encoded["Technology"]["tech"])
    )
)

print("\nIn exported SQLite:")
print(
    sorted(
        truck_tech_names
        & set(db_test["Technology"]["tech"])
    )
)

In db_encoded:
['CO2_TRUCK', 'GSL_TRUCK', 'H2_TRUCK', 'METOH_TRUCK']

In exported SQLite:
['CO2_TRUCK', 'GSL_TRUCK', 'H2_TRUCK', 'METOH_TRUCK']


In [70]:
db_mgmt.update_sqlite(
    OUTPUT_SQLITE_WEAK_PATH,
    db_encoded,
)

In [71]:
db_test = db_mgmt.sqlite_to_dfs(
    OUTPUT_SQLITE_WEAK_PATH
)

print(
    f"Technology: {len(db_test['Technology']):,}"
)

display(
    db_test["Technology"]
    .sort_values("tech")
)

Technology: 16


,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
2,CO2_CAP,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
7,CO2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
12,CO2_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of carbon dioxide by truck,GEO001
0,ELC_GEN,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
5,ELC_TRANS,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
11,GSL_BACKUP,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
10,GSL_DEMAND,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
9,GSL_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
4,GSL_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
14,GSL_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of gasoline by truck,GEO001


In [72]:
# =============================================================================
# Validate exported truck technologies and rows
# =============================================================================

truck_techs = set(truck_tech_specs["tech"])
expected_truck_rows = len(road_links_weak) * len(truck_tech_specs)

assert truck_techs.issubset(set(db_test["Technology"]["tech"]))

assert db_test["Efficiency"]["tech"].isin(truck_techs).sum() == expected_truck_rows
assert db_test["CostVariable"]["tech"].isin(truck_techs).sum() == expected_truck_rows
assert db_test["CostInvest"]["tech"].isin(truck_techs).sum() == expected_truck_rows

print("Truck technology rows validated.")

Truck technology rows validated.


In [73]:
# =============================================================================
# Validate exported node-region references
# =============================================================================

valid_node_regions = set(db_test["Region"]["region"])

for table_name, df in db_test.items():

    if "region" not in df.columns:
        continue

    region_values = df["region"].dropna().astype(str)
    node_regions = region_values[
        ~region_values.str.contains("-", regex=False)
    ]

    invalid_node_regions = sorted(set(node_regions) - valid_node_regions)

    assert not invalid_node_regions, (
        f"{table_name} has invalid node regions: {invalid_node_regions[:10]}"
    )

print("Node-region references validated.")

Node-region references validated.


In [74]:
# =============================================================================
# Validate output tables are empty
# =============================================================================

for table_name, df in db_test.items():
    if table_name.startswith("Output"):
        assert len(df) == 0, f"{table_name} is not empty."

print("Output tables validated.")

Output tables validated.


In [75]:
# =============================================================================
# Validate output tables are empty
# =============================================================================

for table_name, df in db_test.items():
    if table_name.startswith("Output"):
        assert len(df) == 0, f"{table_name} is not empty."

print("Output tables validated.")

Output tables validated.


In [76]:
# =============================================================================
# Notebook 7 final export validation
# =============================================================================

print("Notebook 7 export validated.")
print(f"Output database: {OUTPUT_SQLITE_WEAK_PATH}")

Notebook 7 export validated.
Output database: c:\Users\aviga\Research\repos\temoa_geospace\data_files\processed\schema\CANOE_geospatial_1deg_graph_roads_weak.sqlite


The sqlite db is now constructed. Next Cells will refer to model debugging and db structure.

In [77]:
# =============================================================================
# Validate edge-region endpoints
# =============================================================================

for table_name, df in db_encoded.items():

    if "region" not in df.columns:
        continue

    edge_values = (
        df["region"]
        .dropna()
        .astype(str)
    )

    edge_values = edge_values[
        edge_values.str.contains("-", regex=False)
    ]

    if len(edge_values) == 0:
        continue

    edge_parts = edge_values.str.split("-", n=1, expand=True)

    invalid_edges = edge_values.loc[
        ~(
            edge_parts.iloc[:, 0].isin(valid_node_regions).values
            &
            edge_parts.iloc[:, 1].isin(valid_node_regions).values
        )
    ]

    assert invalid_edges.empty, (
        f"{table_name} has invalid edge endpoints:\n"
        f"{invalid_edges.head()}"
    )

print("Edge-region endpoints validated.")

Edge-region endpoints validated.
